# Spatial Indexes

## Overview
Spatial indexes are what make PostGIS queries fast on large datasets. Without them, every spatial predicate requires a full table scan. With a GIST index, PostGIS uses an R-tree structure to rapidly find candidate geometries by bounding box before checking the exact predicate.

**How GIST works:**
```
Query: ST_Within(site.geom, watershed_polygon)
Step 1: Index scan — find all geoms whose bounding box overlaps watershed bbox  (fast)
Step 2: Recheck   — exact ST_Within test on candidates only                     (precise)
```

**Creating a spatial index (always do this before spatial queries):**
```sql
CREATE INDEX idx_sites_geom    ON monitoring_sites    USING GIST (geom);
CREATE INDEX idx_obs_geom      ON species_observations USING GIST (geom);
CREATE INDEX idx_protected_geom ON protected_areas     USING GIST (geom);
CREATE INDEX idx_watersheds_geom ON watersheds         USING GIST (geom);
```

---

In [ ]:
print("=== Spatial indexes: GIST and BRIN ===")
print()

print("Why spatial indexes are essential:")
print("""
  Without a spatial index, every spatial query requires checking ALL rows:
  ST_Within(site.geom, watershed.geom) on 1M sites x 10K watersheds
  = 10 BILLION geometry comparisons — unusable.

  A GIST index partitions geometries into bounding boxes (R-tree structure).
  The query first filters candidate bounding boxes (fast), then checks exact geometry.
  This reduces comparisons from O(N*M) to O(log N).
""")

index_types = [
    ("GIST",   "R-tree bounding box partitioning",
     "ST_Within, ST_Intersects, ST_DWithin, KNN <->",
     "General-purpose spatial; best for most queries"),
    ("BRIN",   "Block Range INdex — bounding box per block",
     "Very large tables with spatial locality",
     "Tiny; fast to build; approximate (false positives)"),
    ("SP-GIST","Space-partitioning tree (quad-tree, k-d tree)",
     "Certain point-only queries",
     "Less common; specialised use cases"),
]
print(f"  {'Type':<8s}  {'Structure':<38s}  {'Useful for':<34s}  Notes")
print("  " + "-"*100)
for idx, struct, useful, notes in index_types:
    print(f"  {idx:<8s}  {struct:<38s}  {useful:<34s}  {notes}")

print()
print("Creating spatial indexes in PostGIS:")
ddl = [
    ("Basic GIST index:",
     "CREATE INDEX idx_sites_geom ON monitoring_sites USING GIST (geom);"),
    ("Partial index (active sites only):",
     "CREATE INDEX idx_active_sites ON monitoring_sites USING GIST (geom) WHERE active = true;"),
    ("3D geometry index:",
     "CREATE INDEX idx_3d ON elevation_points USING GIST (geom gist_geometry_ops_nd);"),
    ("Geography type (accurate distance on sphere):",
     "CREATE INDEX idx_geo ON sites USING GIST (geom::geography);"),
    ("BRIN for large append-only tables:",
     "CREATE INDEX idx_brin ON sensor_readings USING BRIN (geom);"),
]
for label, sql in ddl:
    print(f"  {label}")
    print(f"    {sql}")
    print()


---
## Reading EXPLAIN ANALYZE for spatial queries

In [ ]:
print("=== EXPLAIN ANALYZE for spatial queries ===")
print()

print("Reading spatial query plans:")
explain_examples = [
    ("Index Scan using idx_sites_geom",
     "Good: spatial index used",
     "Query is efficient; bounding-box filter + exact check"),
    ("Seq Scan on monitoring_sites",
     "Bad: no index",
     "Add GIST index; or query is not sargable (func wrapped geom)"),
    ("Index Cond: (geom && '...'::geometry)",
     "Bounding box filter (&&)",
     "R-tree lookup step; followed by exact predicate recheck"),
    ("Rows Removed by Filter: 99500",
     "High false positive rate",
     "GIST is approximate; many bbox matches but not actual intersects"),
    ("Bitmap Index Scan + Bitmap Heap Scan",
     "Bulk spatial query",
     "Many matching rows; bitmap approach more efficient than index scan"),
]
print(f"  {'Plan node':<46s}  {'Means':<14s}  Implication")
print("  " + "-"*85)
for plan, means, impl in explain_examples:
    print(f"  {plan:<46s}  {means:<14s}  {impl}")

print()
print("Sargability: when spatial indexes are NOT used:")
non_sargable = [
    ("ST_Distance(a, b) < 1000",
     "ST_DWithin(a, b, 1000)",
     "ST_Distance computes exact distance for ALL rows before filtering"),
    ("ST_Buffer(a, 1000) && b",
     "ST_DWithin(a, b, 1000)",
     "Buffer creates geometry per row; DWithin uses index directly"),
    ("ST_Transform(a, 32620) && b",
     "Store data in correct CRS; avoid per-row transform",
     "Transform forces full scan; store projected geometry instead"),
    ("NOT ST_Intersects(a, b)",
     "Use ST_Disjoint(a, b) or LEFT JOIN ... WHERE b IS NULL",
     "NOT prevents index use in most planners"),
]
print(f"  {'Avoid':<44s}  {'Use instead':<36s}  Why")
print("  " + "-"*90)
for avoid, use, why in non_sargable:
    print(f"  {avoid:<44s}  {use:<36s}  {why}")


---
## CLUSTER and geometry simplification

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


print("=== CLUSTER and geometry simplification for performance ===")
print()

from shapely.geometry import Polygon
import geopandas as gpd

# Demonstrate simplification
complex_poly = protected[protected.pa_id=='PA01'].geometry.iloc[0]
# Normally these would be complex multi-vertex polygons
print("Geometry simplification:")
print(f"  Original vertices: {len(complex_poly.exterior.coords)}")

# Douglas-Peucker simplification
simplified = complex_poly.simplify(0.01, preserve_topology=True)
print(f"  After simplify(0.01°): {len(simplified.exterior.coords)} vertices")
simplified2 = complex_poly.simplify(0.1, preserve_topology=True)
print(f"  After simplify(0.1°):  {len(simplified2.exterior.coords)} vertices")

print()
print("PostGIS performance techniques:")
techniques = [
    ("CLUSTER table USING idx_geom",
     "Physically reorder table rows by spatial index — massive speedup for range queries"),
    ("ST_Simplify(geom, tolerance)",
     "Reduce vertex count (Douglas-Peucker); use for display, not analysis"),
    ("ST_SimplifyPreserveTopology(geom, tol)",
     "Simplify without creating invalid geometry (preferred)"),
    ("ST_SnapToGrid(geom, 0.0001)",
     "Snap coordinates to grid; reduces precision; smaller storage"),
    ("geometry vs geography type",
     "geometry: fast, CRS-aware but flat-earth. geography: accurate sphere but slower"),
    ("Partial indexes",
     "CREATE INDEX ... WHERE province='NB' — index only subset of rows"),
    ("Materialized views",
     "Pre-compute expensive spatial joins; refresh on schedule"),
    ("Partitioning",
     "Partition by province/region; spatial queries hit only relevant partition"),
]
for technique, desc in techniques:
    print(f"  {technique:<42s}  {desc}")


---
## Common Pitfalls

**1. Creating index after loading data — forgetting ANALYZE**
After `CREATE INDEX`, PostgreSQL's query planner may not use it until statistics are updated. Always run `ANALYZE monitoring_sites` after creating a spatial index on an existing table so the planner knows the index exists and has correct cardinality estimates.

**2. Spatial index on wrong geometry column**
If your table has both a `geom` and a `geog` (geography) column, an index on `geom` does not help queries filtering on `geog`. Index the column actually used in queries.

**3. Using GIST index for exact equality checks**
GIST indexes do not help `WHERE geom = ST_GeomFromText(...)`. GIST is for spatial operators (`&&`, `ST_Intersects`, `<->`). For exact geometry equality, you need a hash index or a unique constraint on a text WKT column.

**4. Not accounting for GIST false positives**
GIST first filters by bounding box (fast, may return false positives), then the exact predicate is re-evaluated. High false positive rates (many bbox matches that fail exact check) indicate poor spatial locality. CLUSTER the table by the spatial index to improve physical data locality.

**5. Applying function to indexed column — defeating the index**
`WHERE ST_Transform(geom, 32620) && bbox` forces a transform on every row before the index can be applied. Store data in the target CRS, or create a functional index: `CREATE INDEX ON sites USING GIST (ST_Transform(geom, 32620))`.

**6. BRIN index on spatially unsorted data**
BRIN indexes assume spatial locality — nearby rows in the heap have nearby geometry. If rows were inserted in random spatial order, BRIN has no useful structure and returns almost all blocks as candidates. BRIN only works well after a `CLUSTER` by spatial index or for append-only time-series with inherent spatial ordering.


---
*sql_methods_library - Samantha McGarrigle*